# 🤖 Treinamento de Modelos - Random Forest

## Objetivo
Este notebook foca no treinamento e otimização do modelo Random Forest Regressor para previsão de casos de dengue.

## Estratégia
- Divisão temporal dos dados (train/validation/test)
- Otimização de hiperparâmetros com Grid Search
- Análise de importância das features
- Avaliação detalhada do modelo
- Análise de erros por estado

In [ ]:
# Importação das bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import joblib
import warnings

warnings.filterwarnings('ignore')
plt.style.use('default')
sns.set_palette("husl")

print("✅ Bibliotecas carregadas com sucesso!")

In [ ]:
# Carregamento dos dados processados
print("📂 Carregando dados processados...")

df = pd.read_csv('dados_com_features.csv')
X = pd.read_csv('X_features.csv')
y = pd.read_csv('y_target.csv')['Quantidade_Casos']

# Recriar coluna de data para divisão temporal
df['Data'] = pd.to_datetime(df[['Ano', 'Mês']].assign(day=1))

print(f"📊 Dataset: {X.shape[0]:,} registros, {X.shape[1]} features")
print(f"🎯 Target: {y.shape[0]:,} valores")
print(f"📅 Período: {df['Data'].min().strftime('%Y-%m')} até {df['Data'].max().strftime('%Y-%m')}")

In [ ]:
# Divisão temporal dos dados
def split_time_series_data(df, X, y, train_end='2021-12', val_end='2023-12'):
    """
    Divide os dados de forma temporal para evitar data leakage
    """
    # Criar máscara temporal
    train_mask = df['Data'] <= pd.to_datetime(train_end)
    val_mask = (df['Data'] > pd.to_datetime(train_end)) & (df['Data'] <= pd.to_datetime(val_end))
    test_mask = df['Data'] > pd.to_datetime(val_end)

    # Dividir os dados
    X_train = X[train_mask]
    y_train = y[train_mask]

    X_val = X[val_mask]
    y_val = y[val_mask]

    X_test = X[test_mask]
    y_test = y[test_mask]

    return X_train, X_val, X_test, y_train, y_val, y_test, train_mask, val_mask, test_mask

print("📅 Dividindo dados temporalmente...")
X_train, X_val, X_test, y_train, y_val, y_test, train_mask, val_mask, test_mask = split_time_series_data(df, X, y)

print(f"\n📊 Divisão dos dados:")
print(f"   🏋️ Treino: {X_train.shape[0]:,} registros ({train_mask.sum()/len(df)*100:.1f}%)")
print(f"   🎯 Validação: {X_val.shape[0]:,} registros ({val_mask.sum()/len(df)*100:.1f}%)")
print(f"   🧪 Teste: {X_test.shape[0]:,} registros ({test_mask.sum()/len(df)*100:.1f}%)")

# Períodos
print(f"\n📅 Períodos:")
print(f"   Treino: {df[train_mask]['Data'].min().strftime('%Y-%m')} até {df[train_mask]['Data'].max().strftime('%Y-%m')}")
print(f"   Validação: {df[val_mask]['Data'].min().strftime('%Y-%m')} até {df[val_mask]['Data'].max().strftime('%Y-%m')}")
print(f"   Teste: {df[test_mask]['Data'].min().strftime('%Y-%m')} até {df[test_mask]['Data'].max().strftime('%Y-%m')}")

In [ ]:
# Função para calcular métricas
def calculate_metrics(y_true, y_pred, model_name="Modelo"):
    """
    Calcula métricas de regressão
    """
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    # MAPE (Mean Absolute Percentage Error)
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100

    metrics = {
        'Modelo': model_name,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R²': r2,
        'MAPE': mape
    }

    return metrics

# Função para plotar resultados
def plot_predictions(y_true, y_pred, title="Predições vs Real"):
    """
    Plota gráfico de predições vs valores reais
    """
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    # Scatter plot
    axes[0].scatter(y_true, y_pred, alpha=0.6)
    axes[0].plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=2)
    axes[0].set_xlabel('Valores Reais')
    axes[0].set_ylabel('Predições')
    axes[0].set_title(f'{title} - Scatter')

    # Residuos
    residuals = y_pred - y_true
    axes[1].scatter(y_pred, residuals, alpha=0.6)
    axes[1].axhline(y=0, color='r', linestyle='--')
    axes[1].set_xlabel('Predições')
    axes[1].set_ylabel('Resíduos')
    axes[1].set_title(f'{title} - Resíduos')

    plt.tight_layout()
    plt.show()

print("✅ Funções de avaliação definidas!")

In [ ]:
# Modelo baseline - Random Forest simples
print("🌳 Treinando modelo baseline Random Forest...")

# Modelo baseline
rf_baseline = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Treinamento
rf_baseline.fit(X_train, y_train)

# Predições
y_pred_train_baseline = rf_baseline.predict(X_train)
y_pred_val_baseline = rf_baseline.predict(X_val)

# Métricas
metrics_train_baseline = calculate_metrics(y_train, y_pred_train_baseline, "RF Baseline - Treino")
metrics_val_baseline = calculate_metrics(y_val, y_pred_val_baseline, "RF Baseline - Validação")

print("📊 Métricas do modelo baseline:")
print("\n🏋️ Treino:")
for key, value in metrics_train_baseline.items():
    if key != 'Modelo':
        print(f"   {key}: {value:.4f}")

print("\n🎯 Validação:")
for key, value in metrics_val_baseline.items():
    if key != 'Modelo':
        print(f"   {key}: {value:.4f}")

# Visualização
plot_predictions(y_val, y_pred_val_baseline, "Random Forest Baseline - Validação")

In [ ]:
# Otimização de hiperparâmetros
print("🔧 Iniciando otimização de hiperparâmetros...")

# Grid de parâmetros para Random Forest
param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['auto', 'sqrt', 0.3]
}

print(f"🔍 Testando {np.prod([len(v) for v in param_grid_rf.values()])} combinações de hiperparâmetros...")

# Cross-validation temporal
tscv = TimeSeriesSplit(n_splits=3)

# Grid Search
rf_grid = RandomForestRegressor(random_state=42, n_jobs=-1)
grid_search = GridSearchCV(
    rf_grid,
    param_grid_rf,
    cv=tscv,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

# Fit na combinação treino + validação para otimização
X_train_val = pd.concat([X_train, X_val])
y_train_val = pd.concat([y_train, y_val])

grid_search.fit(X_train_val, y_train_val)

print("✅ Otimização concluída!")
print(f"\n🏆 Melhores hiperparâmetros:")
for param, value in grid_search.best_params_.items():
    print(f"   {param}: {value}")

print(f"\n📊 Melhor score (neg_MSE): {grid_search.best_score_:.4f}")

In [ ]:
# Modelo otimizado
print("🚀 Treinando modelo otimizado...")

# Melhor modelo
rf_optimized = grid_search.best_estimator_

# Retreinar nos dados de treino apenas
rf_optimized.fit(X_train, y_train)

# Predições
y_pred_train_opt = rf_optimized.predict(X_train)
y_pred_val_opt = rf_optimized.predict(X_val)
y_pred_test_opt = rf_optimized.predict(X_test)

# Métricas
metrics_train_opt = calculate_metrics(y_train, y_pred_train_opt, "RF Otimizado - Treino")
metrics_val_opt = calculate_metrics(y_val, y_pred_val_opt, "RF Otimizado - Validação")
metrics_test_opt = calculate_metrics(y_test, y_pred_test_opt, "RF Otimizado - Teste")

print("📊 Métricas do modelo otimizado:")
print("\n🏋️ Treino:")
for key, value in metrics_train_opt.items():
    if key != 'Modelo':
        print(f"   {key}: {value:.4f}")

print("\n🎯 Validação:")
for key, value in metrics_val_opt.items():
    if key != 'Modelo':
        print(f"   {key}: {value:.4f}")

print("\n🧪 Teste:")
for key, value in metrics_test_opt.items():
    if key != 'Modelo':
        print(f"   {key}: {value:.4f}")

# Comparação com baseline
print(f"\n📈 Melhoria na validação:")
print(f"   RMSE: {metrics_val_baseline['RMSE']:.4f} → {metrics_val_opt['RMSE']:.4f} ({((metrics_val_baseline['RMSE'] - metrics_val_opt['RMSE'])/metrics_val_baseline['RMSE']*100):+.2f}%)")
print(f"   R²: {metrics_val_baseline['R²']:.4f} → {metrics_val_opt['R²']:.4f} ({((metrics_val_opt['R²'] - metrics_val_baseline['R²'])/metrics_val_baseline['R²']*100):+.2f}%)")

In [ ]:
# Análise de importância das features
print("📊 Analisando importância das features...")

# Importância das features
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_optimized.feature_importances_
}).sort_values('importance', ascending=False)

# Top 20 features mais importantes
top_features = feature_importance.head(20)

plt.figure(figsize=(12, 8))
sns.barplot(data=top_features, y='feature', x='importance')
plt.title('Top 20 Features Mais Importantes - Random Forest')
plt.xlabel('Importância')
plt.tight_layout()
plt.show()

print("🏆 Top 10 features mais importantes:")
for i, (_, row) in enumerate(top_features.head(10).iterrows(), 1):
    print(f"   {i:2d}. {row['feature']}: {row['importance']:.4f}")

# Salvar importância das features
feature_importance.to_csv('feature_importance_rf.csv', index=False)
print("\n💾 Importância das features salva em 'feature_importance_rf.csv'")

In [ ]:
# Análise por estado - dados de teste
print("🗺️ Analisando performance por estado...")

# Criar DataFrame com resultados do teste
df_test = df[test_mask].copy()
df_test['Predicoes'] = y_pred_test_opt
df_test['Residuos'] = df_test['Predicoes'] - df_test['Quantidade de Casos']
df_test['Residuos_Abs'] = np.abs(df_test['Residuos'])

# Métricas por estado
metrics_by_state = []
for state in df_test['COD_UF'].unique():
    state_data = df_test[df_test['COD_UF'] == state]
    if len(state_data) > 0:
        metrics = calculate_metrics(
            state_data['Quantidade de Casos'],
            state_data['Predicoes'],
            state
        )
        metrics['Estado'] = state
        metrics['N_Obs'] = len(state_data)
        metrics_by_state.append(metrics)

metrics_df = pd.DataFrame(metrics_by_state)

# Top e bottom estados por R²
print("\n🏆 Top 5 estados com melhor R²:")
for _, row in metrics_df.nlargest(5, 'R²').iterrows():
    print(f"   {row['Estado']}: R² = {row['R²']:.3f}, RMSE = {row['RMSE']:.2f}")

print("\n🎯 5 estados com menor R²:")
for _, row in metrics_df.nsmallest(5, 'R²').iterrows():
    print(f"   {row['Estado']}: R² = {row['R²']:.3f}, RMSE = {row['RMSE']:.2f}")

# Visualização das métricas por estado
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# R² por estado
axes[0, 0].bar(metrics_df['Estado'], metrics_df['R²'])
axes[0, 0].set_title('R² por Estado')
axes[0, 0].set_ylabel('R²')
axes[0, 0].tick_params(axis='x', rotation=45)

# RMSE por estado
axes[0, 1].bar(metrics_df['Estado'], metrics_df['RMSE'])
axes[0, 1].set_title('RMSE por Estado')
axes[0, 1].set_ylabel('RMSE')
axes[0, 1].tick_params(axis='x', rotation=45)

# MAE por estado
axes[1, 0].bar(metrics_df['Estado'], metrics_df['MAE'])
axes[1, 0].set_title('MAE por Estado')
axes[1, 0].set_ylabel('MAE')
axes[1, 0].tick_params(axis='x', rotation=45)

# MAPE por estado
axes[1, 1].bar(metrics_df['Estado'], metrics_df['MAPE'])
axes[1, 1].set_title('MAPE por Estado (%)')
axes[1, 1].set_ylabel('MAPE (%)')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Salvar métricas por estado
metrics_df.to_csv('metricas_por_estado_rf.csv', index=False)
print("\n💾 Métricas por estado salvas em 'metricas_por_estado_rf.csv'")

In [ ]:
# Visualização das predições no conjunto de teste
plot_predictions(y_test, y_pred_test_opt, "Random Forest Otimizado - Teste")

# Série temporal das predições vs real para alguns estados
estados_exemplo = ['SP', 'MG', 'RJ', 'BA', 'PR']

fig, axes = plt.subplots(len(estados_exemplo), 1, figsize=(15, 3*len(estados_exemplo)))

for i, estado in enumerate(estados_exemplo):
    state_data = df_test[df_test['COD_UF'] == estado].sort_values('Data')

    if len(state_data) > 0:
        axes[i].plot(state_data['Data'], state_data['Quantidade de Casos'],
                    label='Real', marker='o', linewidth=2)
        axes[i].plot(state_data['Data'], state_data['Predicoes'],
                    label='Predição', marker='s', linewidth=2, alpha=0.8)
        axes[i].set_title(f'Predições vs Real - {estado}')
        axes[i].set_ylabel('Casos de Dengue')
        axes[i].legend()
        axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Análise de resíduos detalhada
print("🔍 Análise detalhada de resíduos...")

residuals = y_pred_test_opt - y_test

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Histograma dos resíduos
axes[0, 0].hist(residuals, bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Distribuição dos Resíduos')
axes[0, 0].set_xlabel('Resíduos')
axes[0, 0].set_ylabel('Frequência')

# Q-Q plot
from scipy import stats
stats.probplot(residuals, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title('Q-Q Plot dos Resíduos')

# Resíduos vs predições
axes[1, 0].scatter(y_pred_test_opt, residuals, alpha=0.6)
axes[1, 0].axhline(y=0, color='r', linestyle='--')
axes[1, 0].set_xlabel('Predições')
axes[1, 0].set_ylabel('Resíduos')
axes[1, 0].set_title('Resíduos vs Predições')

# Resíduos absolutos vs predições
axes[1, 1].scatter(y_pred_test_opt, np.abs(residuals), alpha=0.6)
axes[1, 1].set_xlabel('Predições')
axes[1, 1].set_ylabel('|Resíduos|')
axes[1, 1].set_title('|Resíduos| vs Predições')

plt.tight_layout()
plt.show()

# Estatísticas dos resíduos
print(f"\n📊 Estatísticas dos resíduos:")
print(f"   Média: {residuals.mean():.4f}")
print(f"   Desvio Padrão: {residuals.std():.4f}")
print(f"   Mediana: {np.median(residuals):.4f}")
print(f"   MAD (Desvio Absoluto Mediano): {np.median(np.abs(residuals - np.median(residuals))):.4f}")

In [ ]:
# Salvar modelo treinado
print("💾 Salvando modelo treinado...")

# Salvar o modelo
joblib.dump(rf_optimized, 'modelo_random_forest.pkl')
print("✅ Modelo salvo: 'modelo_random_forest.pkl'")

# Salvar hiperparâmetros
with open('hiperparametros_rf.txt', 'w', encoding='utf-8') as f:
    f.write("HIPERPARÂMETROS OTIMIZADOS - RANDOM FOREST\n")
    f.write("="*50 + "\n\n")
    for param, value in grid_search.best_params_.items():
        f.write(f"{param}: {value}\n")

print("✅ Hiperparâmetros salvos: 'hiperparametros_rf.txt'")

# Compilar todas as métricas
all_metrics = [
    metrics_train_baseline, metrics_val_baseline,
    metrics_train_opt, metrics_val_opt, metrics_test_opt
]

metrics_comparison = pd.DataFrame(all_metrics)
metrics_comparison.to_csv('metricas_comparacao_rf.csv', index=False)
print("✅ Comparação de métricas salva: 'metricas_comparacao_rf.csv'")

# Salvar predições do teste
test_predictions = pd.DataFrame({
    'Real': y_test,
    'Predicao': y_pred_test_opt,
    'Residuo': y_pred_test_opt - y_test
})
test_predictions.to_csv('predicoes_teste_rf.csv', index=False)
print("✅ Predições do teste salvas: 'predicoes_teste_rf.csv'")

In [ ]:
# Resumo final
print("🎯 RESUMO - RANDOM FOREST")
print("="*50)
print(f"\n📊 Modelo Final:")
print(f"   Algoritmo: Random Forest Regressor")
print(f"   N_estimators: {rf_optimized.n_estimators}")
print(f"   Max_depth: {rf_optimized.max_depth}")
print(f"   Min_samples_split: {rf_optimized.min_samples_split}")
print(f"   Min_samples_leaf: {rf_optimized.min_samples_leaf}")

print(f"\n📈 Performance (Teste):")
print(f"   RMSE: {metrics_test_opt['RMSE']:.2f}")
print(f"   MAE: {metrics_test_opt['MAE']:.2f}")
print(f"   R²: {metrics_test_opt['R²']:.4f}")
print(f"   MAPE: {metrics_test_opt['MAPE']:.2f}%")

print(f"\n🏆 Top 3 Features Mais Importantes:")
for i, (_, row) in enumerate(top_features.head(3).iterrows(), 1):
    print(f"   {i}. {row['feature']}: {row['importance']:.4f}")

print(f"\n🗺️ Performance por Região:")
print(f"   Melhor estado (R²): {metrics_df.loc[metrics_df['R²'].idxmax(), 'Estado']} ({metrics_df['R²'].max():.3f})")
print(f"   Pior estado (R²): {metrics_df.loc[metrics_df['R²'].idxmin(), 'Estado']} ({metrics_df['R²'].min():.3f})")
print(f"   R² médio: {metrics_df['R²'].mean():.3f} ± {metrics_df['R²'].std():.3f}")

print("\n✅ Random Forest - Treinamento Concluído!")
print("Próximo: Treinamento XGBoost e LightGBM")